# Demo: detección de fuentes (paquete `trust_sources`)

Muestra los dos detectores, la salida estructurada (formato tipo Trust) y el benchmark v0.

In [1]:
import sys; sys.path.insert(0, '..')
from trust_sources import ClassicSourceDetector, LLMSourceDetector
from trust_sources.io_anotaciones import load_double_annotated
arts, xig = load_double_annotated()
print(len(arts), 'artículos doble-anotados')

16 artículos doble-anotados


## Una nota: fuentes según el humano

In [2]:
a = next(x for x in arts if x.index == '105')
print(a.titulo)
print('FUENTES (humano):', a.referenciados)

24M. Schiaretti envió un mensaje por el Día de la Memoria y marcó diferencias con la gestión de Milei 
FUENTES (humano): ['Schiaretti', 'El exgobernador Juan Schiaretti', 'la vicepresidenta Victoria Villarruel', 'Schiaretti', 'Schiaretti']


## Detector clásico: salida estructurada (formato tipo Trust `get_explicit_sources`)

In [3]:
import json
srcs = ClassicSourceDetector().detect(a.cuerpo)
print(json.dumps([s.to_dict() for s in srcs[:1]], ensure_ascii=False, indent=1))

[
 {
  "text": "aseguró que “no fueron 30 mil”.\nSpot. \nEn su posteo, Schiaretti marcó claras diferencias con Milei",
  "start_char": 937,
  "end_char": 1035,
  "length": 98,
  "pattern": "cita_comillas",
  "explicit": true,
  "components": {
   "afirmacion": {
    "text": "“no fueron 30 mil”",
    "start_char": 949,
    "end_char": 967,
    "label": "Afirmacion"
   },
   "conector": {
    "text": "aseguró",
    "start_char": 937,
    "end_char": 944,
    "label": "Conector"
   },
   "referenciado": {
    "text": "Milei",
    "start_char": 1030,
    "end_char": 1035,
    "label": "Referenciado"
   }
  }
 }
]


## Vista v0 (solo referenciados) de cada detector

In [4]:
print('CLASICO:', ClassicSourceDetector().referenciados(a.cuerpo))
import json
cache = json.load(open('cache/llm_sources.json', encoding='utf-8'))
print('LLM (cache):', cache.get(a.index))

CLASICO: ['Milei', 'Schiaretti']
LLM (cache): ['Juan Schiaretti', 'Sonia Torres', 'Abuelas de Plaza de Mayo de Córdoba', 'Victoria Villarruel', 'Alejandra Vigo', 'Hijos']


## Benchmark v0

Correr desde la raíz: `python -m experiments.run_benchmark`.

| Detector | F1 |
|---|---|
| Clásico | 0.26 |
| LLM (Gemini) | 0.56 |
| Techo humano | 0.71 |